# calibrated-typed-decisions: update the card's results section
Replaces the single-run results with the multi-seed and ensemble results. Card only, no data changes. Run once with `PUBLISH = False` first.

## Cell 1 — Install, Drive and Hugging Face login
No GPU needed. The login needs a WRITE token.

In [ ]:
!pip install -q datasets huggingface_hub
from google.colab import drive
drive.mount("/content/drive")
from huggingface_hub import notebook_login
notebook_login()

## Cell 2 — Build the new results section
Reads the files Steps 2 and 3 saved, and writes the section. Nothing is uploaded yet.

In [3]:
import os, re
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import DatasetCard

PUBLISH  = True        # True: upload the updated card (data is not touched)
HF_REPO  = "m-lagnajit/calibrated-typed-decisions"
BASE_DIR = "/content/drive/MyDrive/ctd_student_v2"
RUNS     = ["A", "B", "C", "D", "E", "F"]
TARGETS  = {"A": "one-hot `gold`", "B": "`gold`, label-smoothed (0.1)", "C": "`soft_label`",
            "D": "`0.5 * one_hot(gold) + 0.5 * soft_label`", "E": "`corrected_soft_label`",
            "F": "`0.5 * one_hot(gold) + 0.5 * corrected_soft_label`"}
MEASURES = ["accuracy", "ECE", "log loss"]

def need(name):
    path = f"{BASE_DIR}/{name}"
    assert os.path.exists(path), f"missing results file: {path}"
    return pd.read_csv(path)
per_seed, cons = need("step2_per_seed.csv"), need("step2_consistency.csv")
ens, ens_boot  = need("step3_ensembles.csv"), need("step3_ensemble_bootstrap.csv")
sn_seed, sn_cons = need("step7_snips_per_seed.csv"), need("step7_snips_consistency.csv")
sn_ens, sn_boot  = need("step7_snips_ensembles.csv"), need("step7_snips_ensemble_bootstrap.csv")
sanity = need("step7_rerun_sanity.csv")
n_seeds = per_seed["seed"].nunique()
assert n_seeds >= 3 and set(per_seed["run"]) == set(RUNS) and set(sn_seed["run"]) == set(RUNS), \
    "expected all six runs with three seeds, on test, held-out and SNIPS"
snips_guess = None
if os.path.exists(f"{BASE_DIR}/step6_snips_results.csv"):
    g = pd.read_csv(f"{BASE_DIR}/step6_snips_results.csv")
    g = g[g["run"] == "guess evenly"]
    if len(g):
        snips_guess = (float(g["SNIPS acc"].iloc[0]), float(g["SNIPS log loss"].iloc[0]))

def md_table(header, rows):
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    return "\n".join(out + ["| " + " | ".join(map(str, r)) + " |" for r in rows])
def mr(x):
    return f"{x.mean():.3f} ({x.min():.3f}–{x.max():.3f})"

# ---------- tables ----------
def seed_table(split):
    rows = [[r] + [mr(per_seed[(per_seed.split == split) & (per_seed.run == r)][m]) for m in MEASURES] for r in RUNS]
    return md_table(["run", "accuracy", "ECE", "log loss"], rows)
def ens_table(split):
    sub = ens[ens.split == split].set_index("run")
    return md_table(["run", "accuracy", "ECE", "log loss"],
                    [[r] + [f"{sub.loc[r, f'{m}: ensemble']:.3f}" for m in MEASURES] for r in RUNS])
sn_seed_table = md_table(
    ["run", "SNIPS accuracy", "SNIPS ECE", "SNIPS log loss", "MASSIVE accuracy (seen)", "MASSIVE log loss (seen)"],
    [[r] + [mr(sn_seed[sn_seed.run == r][c]) for c in ["accuracy", "ECE", "log loss",
                                                       "MASSIVE accuracy (seen)", "MASSIVE log loss (seen)"]] for r in RUNS])
sne = sn_ens.set_index("run")
sn_ens_table = md_table(["run", "accuracy", "ECE", "log loss"],
                        [[r] + [f"{sne.loc[r, f'{m}: ensemble']:.3f}" for m in MEASURES] for r in RUNS])

# ---------- what held up under BOTH checks, on all three kinds of test ----------
def direction(v):
    return "better" if "better" in v else ("worse" if "worse" in v else None)
def robust_of(cons_df, boot_df, split):
    eb = boot_df.assign(comparison=boot_df["comparison"].str.replace(" (ensembles)", "", regex=False))
    if "split" in cons_df:
        cons_df = cons_df[cons_df.split == split]
    if "split" in eb:
        eb = eb[eb.split == split]
    b = cons_df.merge(eb, on=["comparison", "measure"], suffixes=("_seeds", "_ens"))
    b["d1"], b["d2"] = b["verdict_seeds"].map(direction), b["verdict_ens"].map(direction)
    r = b[b["d1"].notna() & (b["d1"] == b["d2"])].copy()
    r["cand"] = r["comparison"].str.split(" vs ").str[0]
    r["base"] = r["comparison"].str.split(" vs ").str[1]
    r["split"] = split
    return r[["split", "cand", "base", "measure", "d1"]].rename(columns={"d1": "direction"})
ROB = pd.concat([robust_of(cons, ens_boot, "test"), robust_of(cons, ens_boot, "heldout"),
                 robust_of(sn_cons, sn_boot, "snips")], ignore_index=True)

def ahead(split, run):      # (opponent, measure) where `run` came out reliably ahead
    out = []
    for r in ROB[ROB.split == split].itertuples():
        if r.cand == run and r.direction == "better": out.append((r.base, r.measure))
        if r.base == run and r.direction == "worse":  out.append((r.cand, r.measure))
    return out
def behind(split, run):     # (opponent, measure) where `run` came out reliably behind
    out = []
    for r in ROB[ROB.split == split].itertuples():
        if r.cand == run and r.direction == "worse":  out.append((r.base, r.measure))
        if r.base == run and r.direction == "better": out.append((r.cand, r.measure))
    return out
def and_list(items):
    items = list(items)
    return "none" if not items else items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]
def describe(pairs):        # "A on ECE and log loss; B on log loss"
    by = {}
    for opp, m in pairs:
        by.setdefault(opp, []).append(m)
    return "; ".join(f"{o} on {and_list(ms)}" for o, ms in sorted(by.items())) or "none"
def robust_lines(split):
    return "\n".join(f"- {r.cand} {r.direction} than {r.base} on {r.measure}"
                     for r in ROB[ROB.split == split].itertuples()) or "- (none)"

# ---------- the facts behind each recommendation (checked; the cell stops if any fails) ----------
ens_ll = {s: ens[ens.split == s].set_index("run")["log loss: ensemble"] for s in ("test", "heldout")}
ens_ll["snips"] = sne["log loss: ensemble"]
ens_ece = {s: ens[ens.split == s].set_index("run")["ECE: ensemble"] for s in ("test", "heldout")}
ens_ece["snips"] = sne["ECE: ensemble"]
best = {s: ens_ll[s].idxmin() for s in ens_ll}

assert not behind("test", "F"), f"something is reliably better than F on test: {behind('test', 'F')}"
af_pairs = [p for p in ahead("test", "F") + behind("test", "F") if p[0] == "A"]
a_tied = not af_pairs and not [p for p in ahead("test", "A") + behind("test", "A") if p[0] == "F"]
assert best["snips"] == "E", f"E no longer has the best SNIPS log loss (best: {best['snips']})"
assert not behind("snips", "E"), f"something is reliably better than E on SNIPS: {behind('snips', 'E')}"
assert best["heldout"] == "F", f"F no longer has the best held-out log loss (best: {best['heldout']})"
assert not behind("heldout", "F"), f"something is reliably better than F on held-out: {behind('heldout', 'F')}"
f_vs_de_far = [p for p in ahead("heldout", "F") if p[0] in ("D", "E")]
c_behind_test = behind("test", "C")

gold_h = pd.Series(load_dataset(HF_REPO, split="heldout")["gold"])
n_test = len(load_dataset(HF_REPO, split="test"))
freq = gold_h.value_counts(normalize=True)
majority, base_ll = float(freq.iloc[0]), float(-(freq * np.log(freq)).sum())
luck = per_seed[per_seed.split == "heldout"].groupby("run")["accuracy"].agg(lambda x: x.max() - x.min())
within = int(sanity["gap within normal seed spread"].astype(str).str.lower().eq("true").sum())
agree_lo, agree_hi = float(sanity["same top pick"].min()), float(sanity["same top pick"].max())
print("best ensemble log loss:", best, "| A tied with F on test:", a_tied)
print("E ahead on SNIPS:", describe(ahead("snips", "E")), "| F ahead on held-out:", describe(ahead("heldout", "F")))

SECTION = f"""## Which training target to use

We trained ModernBERT-base students on the train split with {len(RUNS)} different targets, each with
**{n_seeds} random seeds** (3 epochs; best epoch chosen on validation log loss). Every student gets
**2 temperature dials** fitted on validation: one for choice questions, and one plus a yes/no shift for
yes/no questions. (Cross-fitting on validation preferred this to one dial per source and type, and to no dial.)

{md_table(["run", "target"], [[r, TARGETS[r]] for r in RUNS])}

We tested every student three ways:

| test | what's new to the student | questions |
|---|---|---|
| **test split** | nothing: the same six sources | {n_test:,} |
| **SNIPS** | a new label set of the same kind (voice-assistant intents) | 1,050 |
| **held-out PubMedQA** | a new domain *and* a new kind of judgment (does the evidence support yes / no / maybe?) | {len(gold_h):,} |

### 1. Familiar data: the test split

Single students, mean over {n_seeds} seeds (lowest–highest):

{seed_table("test")}

Ensembles (each run's {n_seeds} calibrated students averaged):

{ens_table("test")}

### 2. A new label set of the same kind: SNIPS

[SNIPS](https://github.com/sonos/nlu-benchmark) (2017 custom intents; CC0-1.0; please cite Coucke et al., 2018,
*Snips Voice Platform*) is **not part of this dataset**: we used it for evaluation only. We built 1,050 questions
from it, 150 per intent, in the same format as this dataset (descriptions, 3–7 options, about 30% yes/no). None of
its requests appear word-for-word in our training data. Seeds 1 and 2 here come from students retrained with the
same recipe and seeds (their weights hadn't been kept); on the test split they agreed with the original students
on {agree_lo:.1%}–{agree_hi:.1%} of top picks, and {within} of {len(sanity)} fell within the original three-seed range.

Single students, mean over {n_seeds} seeds, with MASSIVE (a seen dataset of the same kind) alongside:

{sn_seed_table}

Ensembles:

{sn_ens_table}
{"" if snips_guess is None else f'''
Guessing evenly among the shown options would score {snips_guess[0]:.3f} accuracy and {snips_guess[1]:.3f} log loss.
'''}
### 3. A new kind of judgment: held-out PubMedQA

Single students, mean over {n_seeds} seeds:

{seed_table("heldout")}

Ensembles:

{ens_table("heldout")}

Always answering "{freq.index[0]}" scores {majority:.3f} accuracy; always predicting the label frequencies
gives a log loss of {base_ll:.3f}.

### What held up under both checks

Listed only if it held in **every seed** *and* in a paired bootstrap over questions for the ensembles.

**Test split:**
{robust_lines("test")}

**SNIPS:**
{robust_lines("snips")}

**Held-out PubMedQA:**
{robust_lines("heldout")}

### In short: which target, for what

- **No single target wins everywhere.** Pulling the target toward the one-hot `gold` helps on familiar data and
  hurts on new label sets; the right choice depends on where the student will be used.
- **For data like the six training sources: F.** No target was reliably better than F on the test split.
  {"A with a dial is statistically tied with it" if a_tied else "A with a dial is close"}
  (ensemble log loss {ens_ll['test']['A']:.3f} vs {ens_ll['test']['F']:.3f}).
- **For a new label set of the same kind: E.** On SNIPS, E's ensemble had the lowest log loss
  ({ens_ll['snips']['E']:.3f}). Under both checks it beat {describe(ahead("snips", "E"))}, and nothing was
  reliably better than it. (On SNIPS, E was compared head-to-head with A, B and F only.)
- **For a new kind of judgment, no target is reliably good.** F's ensemble had the best held-out log loss
  ({ens_ll['heldout']['F']:.3f}, against {base_ll:.3f} for predicting the label frequencies) and beat
  {describe(ahead("heldout", "F"))} under both checks; {"against D and E the difference wasn't settled"
  if not f_vs_de_far else "it also beat " + describe(f_vs_de_far)}.
- **Calibration is good where the skill exists.** Ensemble ECE for F: {ens_ece['test']['F']:.3f} on test,
  {ens_ece['snips']['F']:.3f} on SNIPS; for E: {ens_ece['test']['E']:.3f} and {ens_ece['snips']['E']:.3f}. On
  PubMedQA the students are fairly honest (F: {ens_ece['heldout']['F']:.3f}) but know little: the gap there is
  skill, not honesty.
- **`soft_label` alone (C) is the weakest on familiar data**{"" if not c_behind_test else
  " (reliably behind " + describe(c_behind_test) + ")"}, because it inherits the teacher's mistakes.
- **Judge targets on several seeds.** Retraining alone moved one run's held-out accuracy by
  {luck.max():.3f} (run {luck.idxmax()}).

"""

card = DatasetCard.load(HF_REPO)
text = card.text
print("headings on the current card:")
for h in [l for l in text.splitlines() if l.startswith("## ")]:
    print("  ", h)
START = "## Which training target to use"
if START in text:
    start = text.index(START)
    nxt = re.search(r"\n## ", text[start + len(START):])
    end = start + len(START) + nxt.start() + 1 if nxt else len(text)
    NEW_TEXT = text[:start] + SECTION + text[end:]
    print("\n-> replacing the existing results section")
else:
    anchor = next((h for h in ["## Teacher calibration on the test split", "## Sources", "## Limitations"]
                   if h in text), None)
    i = text.index(anchor) if anchor else len(text)
    NEW_TEXT = text[:i] + SECTION + text[i:]
    print(f"\n-> no results section yet: inserting it before {anchor!r}")
assert NEW_TEXT.count(START) == 1, "the card would end up with two results sections; nothing changed"
print("\n" + SECTION)
print(f"(card: {len(text)} -> {len(NEW_TEXT)} characters; only this section changes)")

best ensemble log loss: {'test': 'A', 'heldout': 'F', 'snips': 'E'} | A tied with F on test: True
E ahead on SNIPS: A on ECE and log loss; B on accuracy, ECE and log loss; F on log loss | F ahead on held-out: A on ECE and log loss; B on log loss
headings on the current card:
   ## What changed in v2.1
   ## What changed from v1
   ## Splits
   ## Fields
   ## Which training target to use
   ## Teacher calibration on the test split
   ## Sources
   ## Limitations

-> replacing the existing results section

## Which training target to use

We trained ModernBERT-base students on the train split with 6 different targets, each with
**3 random seeds** (3 epochs; best epoch chosen on validation log loss). Every student gets
**2 temperature dials** fitted on validation: one for choice questions, and one plus a yes/no shift for
yes/no questions. (Cross-fitting on validation preferred this to one dial per source and type, and to no dial.)

| run | target |
|---|---|
| A | one-hot `gold` |
| B | 

## Cell 3 — Publish the card
Uploads only the card. The data stays exactly as v2.1.

In [4]:
if not PUBLISH:
    print("Dry run: nothing uploaded. Read the section above, then set PUBLISH = True in Cell 2 and rerun Cells 2 and 3.")
else:
    card.text = NEW_TEXT
    card.push_to_hub(HF_REPO, commit_message="Card: multi-seed and ensemble results for training targets")
    print(f"Card updated: https://huggingface.co/datasets/{HF_REPO}")

Card updated: https://huggingface.co/datasets/m-lagnajit/calibrated-typed-decisions
